# Case study: loading the Berlin listings with keys and constraints

*Session 3, extras · Author: course team · Licence: CC BY 4.0*

**Why.** The course database `airbnb.sqlite` was built quickly with `to_sql`: only `listings` has a primary key,
and nothing checks the values. A team database should be stricter: loaded the same way every time and protected by
rules that every member's code must respect. This notebook builds a second, strict database from the same Parquet
files, schema first, and documents every decision the rules force us to make. It is the long version of the
section "Creating tables and loading data" of the block 1 lecture, and a model for the team project's loader.

A reproducible **loading script** always does the same steps in the same order:

1. create the tables with their keys and constraints ([`../sql/01-schema.sql`](../sql/01-schema.sql));
2. switch foreign keys on (`PRAGMA foreign_keys = ON`; SQLite checks them only then);
3. load the parent table (`listings`) before the child tables (`calendar`, `reviews`);
4. check the counts against the source files and write down every row that was changed or left out.

The strict database is written to a temporary folder, not into the repository.

In [1]:
import sqlite3
import tempfile
from pathlib import Path

import pandas as pd

# find the repository root (the folder that contains case-study/), wherever the notebook runs
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir())
DATA = ROOT / "case-study" / "data" / "airbnb"
assert (DATA / "listings.parquet").exists(), "run case-study/prepare_airbnb.py first"
SCHEMA = (ROOT / "sessions" / "03-sql" / "workbooks" / "sql" / "01-schema.sql").read_text()

DB = Path(tempfile.gettempdir()) / "airbnb_strict.sqlite"   # a temporary folder, outside the repository

## 1. The schema

Read the schema first: which columns are keys, which rules are checks? `date(x)` returns NULL for text that is not a
valid date, so `CHECK (date IS date(date))` accepts only real dates in the form `'YYYY-MM-DD'`.

> **Note.** A `CHECK` rejects a row only when its condition is *false*; an *unknown* (NULL) result passes. With
> `CHECK (date = date(date))`, the text `'2009-13-01'` gives `'2009-13-01' = NULL`, which is unknown, and the row
> would be accepted. `IS` compares like `=` but treats NULL as an ordinary value (`NULL IS NULL` is true), so the
> rule cannot be bypassed this way.

In [2]:
print(SCHEMA)

-- Schema of a course database for the Berlin Airbnb case study, with keys and constraints (SQLite).
-- Used by extras/01-case-study-loading-with-constraints.ipynb: run it on an empty database, switch
-- foreign keys on (PRAGMA foreign_keys = ON), then load the parent table (listings) before the child
-- tables (calendar, reviews). Dates are text 'YYYY-MM-DD'; available is 1 (available) or 0.
-- Author: course team, licence CC BY 4.0

DROP TABLE IF EXISTS calendar;
DROP TABLE IF EXISTS reviews;
DROP TABLE IF EXISTS listings;

CREATE TABLE listings (                                   -- one row per listing of the snapshot
    id                   INTEGER PRIMARY KEY,
    host_id              INTEGER NOT NULL,
    district             TEXT NOT NULL,
    neighbourhood        TEXT NOT NULL,
    latitude             REAL NOT NULL CHECK (latitude BETWEEN 52.33 AND 52.68),
    longitude            REAL NOT NULL CHECK (longitude BETWEEN 13.08 AND 13.77),
    room_type            TEXT NOT NULL


## 2. A first attempt, exactly as the files are

The loader creates the tables and tries to insert the listings without any change. Dates become text and
true/false becomes 1/0, as SQLite expects. `with con:` makes the insert one transaction: if one row fails, no row
is stored.

In [3]:
LISTING_COLS = ["id", "host_id", "district", "neighbourhood", "latitude", "longitude", "room_type", "accommodates",
                "bedrooms", "price", "minimum_nights", "maximum_nights", "availability_365", "number_of_reviews",
                "first_review", "last_review", "review_scores_rating", "license_status"]
SENTINEL = 2_147_483_647          # the largest 32-bit integer


def fresh_database() -> sqlite3.Connection:
    """Create an empty database with the schema and foreign keys switched on."""
    DB.unlink(missing_ok=True)
    con = sqlite3.connect(DB)
    con.execute("PRAGMA foreign_keys = ON")
    con.executescript(SCHEMA)
    return con


def insert(con: sqlite3.Connection, table: str, frame: pd.DataFrame) -> None:
    """Insert all rows of a DataFrame in one transaction (NaN becomes NULL)."""
    rows = frame.astype(object).where(frame.notna(), None).itertuples(index=False, name=None)
    placeholders = ", ".join("?" for _ in frame.columns)
    with con:                                           # commit at the end, roll back on error
        con.executemany(f"INSERT INTO {table} ({', '.join(frame.columns)}) VALUES ({placeholders})", rows)


listings = pd.read_parquet(DATA / "listings.parquet", columns=LISTING_COLS)
for col in ["first_review", "last_review"]:
    listings[col] = listings[col].dt.strftime("%Y-%m-%d")

con = fresh_database()
try:
    insert(con, "listings", listings)
except sqlite3.IntegrityError as error:
    print("rejected:", error)
print("listings stored:", con.execute("SELECT COUNT(*) FROM listings").fetchone()[0])

rejected: CHECK constraint failed: maximum_nights < 2147483647
listings stored: 0


The database refuses: two listings carry the maximum stay 2,147,483,647, the largest 32-bit integer, which is a
software default and not a stay (Session 4 finds this placeholder too). Because the insert was one transaction, not
a single listing was stored.

## 3. Decisions, then the load

Each change to the data is a **decision**, and each decision is written down:

1. `maximum_nights = 2,147,483,647` becomes NULL (2 listings).
2. Calendar and review rows of listings that are not in the listings table are not loaded, but counted: the foreign
   key would reject them, and Inside Airbnb scraped the files separately, so these listings went offline between two
   scrapes (block 1).

In [4]:
decisions = []
con = fresh_database()

n_sentinel = int(listings["maximum_nights"].eq(SENTINEL).sum())
listings["maximum_nights"] = listings["maximum_nights"].where(listings["maximum_nights"] != SENTINEL)
decisions.append(("listings", "maximum_nights = 2,147,483,647 set to NULL", n_sentinel))
insert(con, "listings", listings)

calendar = pd.read_parquet(DATA / "calendar.parquet", columns=["listing_id", "date", "available"])
calendar["date"] = calendar["date"].dt.strftime("%Y-%m-%d")
calendar["available"] = calendar["available"].astype(int)
try:                                                    # first try: all rows
    insert(con, "calendar", calendar)
except sqlite3.IntegrityError as error:
    print("calendar rejected:", error)

reviews = pd.read_parquet(DATA / "reviews_monthly.parquet")
reviews["month"] = reviews["month"].dt.strftime("%Y-%m-%d")

known = set(listings["id"])
for table, frame in [("calendar", calendar), ("reviews", reviews)]:
    keep = frame["listing_id"].isin(known)
    decisions.append((table, "rows of listings not in listings (not loaded)", int((~keep).sum())))
    insert(con, table, frame[keep])

pd.DataFrame(decisions, columns=["table", "decision", "rows"])

calendar rejected: FOREIGN KEY constraint failed


,table,decision,rows
0,listings,"maximum_nights = 2,147,483,647 set to NULL",2
1,calendar,rows of listings not in listings (not loaded),28835
2,reviews,rows of listings not in listings (not loaded),499


In [5]:
# check the counts against the source files
counts = pd.read_sql_query("""
    SELECT 'listings' AS table_name, COUNT(*) AS n FROM listings
    UNION ALL SELECT 'calendar', COUNT(*) FROM calendar
    UNION ALL SELECT 'reviews', COUNT(*) FROM reviews
""", con)
counts["in_file"] = [len(listings), len(calendar), len(reviews)]
counts.assign(not_loaded=lambda d: d["in_file"] - d["n"])

,table_name,n,in_file,not_loaded
0,listings,12776,12776,0
1,calendar,4663240,4692075,28835
2,reviews,226058,226557,499


The differences are exactly the rows we decided not to load, and we know which listings they belong to.

## 4. The constraints at work

Try to insert invalid rows. The database rejects them, and the error message says which kind of rule failed.

In [6]:
some_id = int(listings["id"].iloc[0])
COLS = ("id, host_id, district, neighbourhood, latitude, longitude, room_type, accommodates, availability_365, "
        "number_of_reviews, license_status")
bad_rows = {
    "unknown room type": f"INSERT INTO listings ({COLS}) VALUES "
                         "(1, 1, 'Mitte', 'Alexanderplatz', 52.52, 13.41, 'Castle', 2, 0, 0, 'missing')",
    "outside Berlin": f"INSERT INTO listings ({COLS}) VALUES "
                      "(2, 1, 'Mitte', 'Alexanderplatz', 48.14, 11.58, 'Private room', 2, 0, 0, 'missing')",
    "unknown listing": "INSERT INTO calendar VALUES (3, '2026-07-01', 1)",
    "same night twice": f"INSERT INTO calendar SELECT listing_id, date, available FROM calendar "
                        f"WHERE listing_id = {some_id} LIMIT 1",
    "not a date": f"INSERT INTO reviews VALUES ({some_id}, '2009-13-01', 1)",
    "zero reviews row": f"INSERT INTO reviews VALUES ({some_id}, '2009-01-01', 0)",
}
for name, statement in bad_rows.items():
    try:
        with con:
            con.execute(statement)
        print(f"{name:18s} accepted (!)")
    except sqlite3.IntegrityError as error:
        print(f"{name:18s} rejected: {str(error)[:90]}")

unknown room type  rejected: CHECK constraint failed: room_type IN ('Entire home/apt', 'Private room', 'Hotel room', 'S
outside Berlin     rejected: CHECK constraint failed: latitude BETWEEN 52.33 AND 52.68
unknown listing    rejected: FOREIGN KEY constraint failed
same night twice   rejected: UNIQUE constraint failed: calendar.listing_id, calendar.date
not a date         rejected: CHECK constraint failed: month IS date(month, 'start of month')
zero reviews row   rejected: CHECK constraint failed: n_reviews > 0


## 5. What the database contains

A few facts computed from the database, together with the loading decisions. A team keeps such a summary next to its
project data, with the source, the licence and the known limitations, so that a new reader can judge what the data
can and cannot answer.

In [7]:
facts = pd.read_sql_query("""
    SELECT COUNT(*) AS n_listings,
           COUNT(DISTINCT host_id) AS n_hosts,
           COUNT(DISTINCT district) AS n_districts,
           COUNT(DISTINCT neighbourhood) AS n_neighbourhoods,
           ROUND(AVG(price IS NULL), 3) AS share_price_missing,
           ROUND(AVG(minimum_nights >= 28), 3) AS share_medium_term,
           ROUND(AVG(number_of_reviews = 0), 3) AS share_without_review,
           ROUND(AVG(license_status = 'registration number'), 3) AS share_registered,
           (SELECT MIN(month) FROM reviews) AS first_review_month,
           (SELECT MIN(date) FROM calendar) AS calendar_from,
           (SELECT MAX(date) FROM calendar) AS calendar_until
    FROM listings
""", con).T.rename(columns={0: "value"})
for table, decision, rows in decisions:
    facts.loc[f"{table}: {decision}"] = rows
con.close()
DB.unlink()                                             # remove the temporary database
facts

,value
n_listings,12776
n_hosts,8182
n_districts,12
n_neighbourhoods,138
share_price_missing,0.339
share_medium_term,0.351
share_without_review,0.201
share_registered,0.351
first_review_month,2009-06-01
calendar_from,2026-06-26


**Discussion.** Which limitations should a reader of an analysis based on these tables know? For example:

- the reviews are those of listings still online in June 2026: deleted listings took their reviews with them
  (**survivorship**), so the growth since 2015 overstates the growth of the market;
- a night that is not available is booked *or* blocked: the calendar does not measure demand;
- prices are asking prices for the next free night, a third of the listings show none, and medium-term listings
  have a different kind of price;
- locations are shifted by up to about 150 m by Airbnb to protect the hosts (Inside Airbnb, *Data assumptions*).

> **Important.** Inside Airbnb publishes its data under the Creative Commons Attribution 4.0 licence (CC BY 4.0):
> reuse is allowed with attribution. The data are scraped from public listing pages and contain personal data, which
> `case-study/prepare_airbnb.py` removes (host names, review texts, names typed into the registration field).
> Document both the licence and these steps with the data, and report results in aggregate, never naming hosts.